In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


# 1) Load data
df = pd.read_csv("Loan.csv")

# Remove approved loans for accuracy
y = df["LoanApproved"].astype(int)
X = df.drop(columns=["LoanApproved"])


# 2) Time-based train/test split
# ApplicationDate: to_datetime ApplicationDate for preventing the data error
X = X.copy()
X["ApplicationDate"] = pd.to_datetime(X["ApplicationDate"], errors="coerce")

# Filter out the NaT date
mask_date = X["ApplicationDate"].notna()
X = X[mask_date].copy()
y = y[mask_date].copy()

# Time-based split. Use the early 80% for train, the late 20% for test
cutoff = X["ApplicationDate"].quantile(0.8)
train_idx = X["ApplicationDate"] <= cutoff
test_idx  = X["ApplicationDate"] > cutoff

X_train, y_train = X[train_idx].copy(), y[train_idx].copy()
X_test,  y_test  = X[test_idx].copy(),  y[test_idx].copy()

X_train

,ApplicationDate,Age,AnnualIncome,CreditScore,EmploymentStatus,EducationLevel,Experience,LoanAmount,LoanDuration,MaritalStatus,...,TotalLiabilities,MonthlyIncome,UtilityBillsPaymentHistory,JobTenure,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio,RiskScore
0,2018-01-01,45,39948,617,Employed,Master,22,13152,48,Married,...,19183,3329.000000,0.724972,11,126928,0.199652,0.227590,419.805992,0.181077,49.0
1,2018-01-02,38,39709,628,Employed,Associate,15,26045,48,Single,...,9595,3309.083333,0.935132,3,43609,0.207045,0.201077,794.054238,0.389852,52.0
2,2018-01-03,47,40724,570,Employed,Bachelor,26,17627,36,Married,...,128874,3393.666667,0.872241,6,5205,0.217627,0.212548,666.406688,0.462157,52.0
3,2018-01-04,58,69084,545,Employed,High School,34,37898,96,Single,...,5370,5757.000000,0.896155,5,99452,0.300398,0.300911,1047.506980,0.313098,54.0
4,2018-01-05,37,103264,594,Employed,Associate,17,9184,36,Married,...,17286,8605.333333,0.941369,5,227019,0.197184,0.175990,330.179140,0.070210,36.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,2061-10-17,45,67286,588,Employed,High School,25,15044,24,Single,...,15012,5607.166667,0.882297,2,5021,0.196044,0.224378,783.711346,0.267998,59.0
15996,2061-10-18,54,142654,670,Employed,Doctorate,32,40602,72,Single,...,5279,11887.833333,0.761932,7,74399,0.220602,0.263843,1128.442090,0.126890,37.6
15997,2061-10-19,36,88971,613,Employed,Doctorate,13,12902,48,Married,...,8521,7414.250000,0.916456,3,160280,0.201402,0.235347,417.317284,0.123993,36.0
15998,2061-10-20,47,43851,561,Employed,Bachelor,23,17634,36,Single,...,159961,3654.250000,0.803127,5,4807,0.222134,0.215562,669.408951,0.396089,62.0


In [10]:
# Drop the date for avoiding seasonal variation
X_train = X_train.drop(columns=["ApplicationDate"])
X_test  = X_test.drop(columns=["ApplicationDate"])


print("Train:", X_train.shape, "Test:", X_test.shape) #Features
print("Train approval rate:", y_train.mean(), "Test approval rate:", y_test.mean()) #Outcome

Train: (16000, 34) Test: (4000, 34)
Train approval rate: 0.23925 Test approval rate: 0.238


In [15]:
# 3) Build model -> p_approve
cat_cols = X_train.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols), #OneHotEncoder for turning object to number
        ("num", "passthrough", num_cols),
    ]
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000)) #LogisticRegression for p_approve score
])

model.fit(X_train, y_train)

# score on test： p_approve
p_approve = model.predict_proba(X_test)[:, 1]

test = X_test.copy()
test["LoanApproved"] = y_test.values
test["p_approve"] = p_approve


# 4) Define payout proxy (Risk-threshold proxy)
# Here allows me to adjust the boaderline for different situations
RISK_TH = 60
DTI_TH  = 0.55

test["payout_proxy"] = (
    (test["LoanApproved"] == 1) & 
    (test["RiskScore"] <= RISK_TH) & #The lowest the better (Risk-threshold)
    (test["TotalDebtToIncomeRatio"] <= DTI_TH) &  #The lowest the better (Total Debt To Income Ratio)
    (test["BankruptcyHistory"] == 0)).astype(int)


# 5) Define policies A/B (Top20 vs Top40) -> "Submit"
def apply_policy(scored_df: pd.DataFrame, top_pct: float) -> pd.DataFrame:
    """Top X% by p_approve => submit=1"""
    d = scored_df.sort_values("p_approve", ascending=False).copy()
    n = int(len(d) * top_pct)
    d["submit"] = 0
    d.loc[d.index[:n], "submit"] = 1
    return d

def get_rank_slice(scored_df: pd.DataFrame, low_pct: float, high_pct: float) -> pd.DataFrame:
    """Return rows ranked between low_pct and high_pct by p_approve (e.g., 20%-40%)."""
    d = scored_df.sort_values("p_approve", ascending=False).copy()
    n_low = int(len(d) * low_pct)
    n_high = int(len(d) * high_pct)
    slice_df = d.iloc[n_low:n_high].copy()
    slice_df["submit"] = 1  
    return slice_df

A = apply_policy(test, 0.20)          # Top20
B = apply_policy(test, 0.40)          # Top40
C = get_rank_slice(test, 0.20, 0.40)  # 20%-40%


# 6) Evaluate funnel metrics
def eval_funnel(d: pd.DataFrame):
    submitted = d[d["submit"] == 1]
    submit_n = len(submitted)

    grants = submitted["LoanApproved"].sum()
    payout = submitted["payout_proxy"].sum()

    grant_rate = grants / submit_n if submit_n else 0.0
    payout_per_grant = payout / grants if grants else 0.0
    e2e_payout_rate = payout / submit_n if submit_n else 0.0

    return {
        "Submit": submit_n,
        "Grants": int(grants),
        "Grant rate": grant_rate,
        "Payout(proxy)": int(payout),
        "Payout per grant": payout_per_grant,
        "End-to-end payout rate": e2e_payout_rate,
        # Guardrails： Check if policy (B) lower the average rate
        "Avg CreditScore (submit)": submitted["CreditScore"].mean() if submit_n else np.nan,
        "Avg RiskScore (submit)": submitted["RiskScore"].mean() if submit_n else np.nan,
        "Avg Total DTI (submit)": submitted["TotalDebtToIncomeRatio"].mean() if submit_n else np.nan,
    }

resA = eval_funnel(A)
resB = eval_funnel(B)
resC = eval_funnel(C)

summary = pd.DataFrame(
    [resA, resB, resC],
    index=["Policy A (Top20%)", "Policy B (Top40%)", "Policy C (20% to 40%)"]
)

summary

C:\Users\s0028\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Submit,Grants,Grant rate,Payout(proxy),Payout per grant,End-to-end payout rate,Avg CreditScore (submit),Avg RiskScore (submit),Avg Total DTI (submit)
Policy A (Top20%),800,740,0.92500,716,0.967568,0.89500,588.898750,40.66075,0.146758
Policy B (Top40%),1600,942,0.58875,918,0.974522,0.57375,585.844375,44.97825,0.202114
Policy C (20% to 40%),800,202,0.25250,202,1.000000,0.25250,582.790000,49.29575,0.257470
